# Fiche 5 — Créer, charger, vérifier

| | |
|---|---|
| 🎯 **Objectif** | Charger des données dans une base et prouver qu'aucune ligne n'a été perdue ou déformée |
| ⏱️ **Durée** | 45 min |
| 🧰 **Prérequis** | Fiches 1 à 3 |
| 🔗 **Dans le projet** | Partie 2 (création de la base, chargement) |

> **L'idée clé.** Un chargement n'est pas fini quand l'import « a marché » : il est fini quand on a **prouvé** que la base contient exactement ce que contenaient les fichiers.

In [1]:
import sqlite3

import pandas as pd

# Une base « jetable » en mémoire : on peut tout casser, rien n'est écrit sur le disque.
# isolation_level=None : chaque instruction est validée aussitôt, comme dans SQLite Studio.
base = sqlite3.connect(":memory:", isolation_level=None)


def sql(requete: str) -> pd.DataFrame | None:
    """Exécute du SQL (une ou plusieurs instructions) et affiche le résultat s'il y en a un."""
    if requete.strip().rstrip(";").count(";"):
        base.executescript(requete)
        return None
    curseur = base.execute(requete)
    if curseur.description:
        return pd.DataFrame(curseur.fetchall(), columns=[c[0] for c in curseur.description])
    return None

sql("""
CREATE TABLE vente (
    id_vente        INTEGER PRIMARY KEY,
    date_vente      TEXT NOT NULL,
    valeur_fonciere REAL
);
""")

## 1. ⚠️ Piège : le vide n'est pas « rien »

Selon l'outil et ses options d'import, une cellule vide d'un CSV peut arriver dans la base comme un **texte vide**
`''` au lieu de `NULL`. Simulons-le, la 3ᵉ vente n'a pas de prix :

In [2]:
sql("""
INSERT INTO vente VALUES (1, '2020-01-02', 165000), (2, '2020-01-03', 229500),
                         (3, '2020-02-10', ''),     (4, '2020-03-01', 98000);
""")
sql("SELECT AVG(valeur_fonciere) AS moyenne, COUNT(valeur_fonciere) AS valeurs_comptees FROM vente")

,moyenne,valeurs_comptees
0,123125.0,4


À coller dans SQLite Studio :
```sql
SELECT AVG(valeur_fonciere) AS moyenne, COUNT(valeur_fonciere) AS valeurs_comptees FROM vente;
UPDATE vente SET valeur_fonciere = NULL WHERE valeur_fonciere = '';
SELECT AVG(valeur_fonciere) AS moyenne, COUNT(valeur_fonciere) AS valeurs_comptees, COUNT(*) AS lignes FROM vente;
```

La moyenne vaut 123 125 au lieu de 164 167 : le texte vide a été compté comme **0**, et `COUNT` le compte comme une
valeur. Après correction en `NULL` :

In [3]:
sql("UPDATE vente SET valeur_fonciere = NULL WHERE valeur_fonciere = ''")
sql("SELECT AVG(valeur_fonciere) AS moyenne, COUNT(valeur_fonciere) AS valeurs_comptees, COUNT(*) AS lignes FROM vente")

,moyenne,valeurs_comptees,lignes
0,164166.666667,3,4


`COUNT(*)` compte les **lignes** (4) ; `COUNT(colonne)` compte les **valeurs renseignées** (3). Leur différence est
le nombre de valeurs manquantes.

## 2. Prouver le chargement : compter des deux côtés

Dans ton tableur, la dernière ligne du CSV moins la ligne d'en-tête donne le nombre de lignes à charger. Dans
SQLite Studio, `SELECT COUNT(*) FROM vente;` donne le nombre de lignes chargées. Les deux doivent être égaux.
Une capture de ces `COUNT(*)` est la preuve attendue dans la présentation.

## 3. ⚠️ Piège : les colonnes dans le mauvais ordre

L'import associe les colonnes **par position**, pas par nom. Si le CSV met le prix avant la date :

In [4]:
sql("INSERT INTO vente VALUES (5, 120000, '2020-04-02')")   # prix et date inversés
sql("SELECT id_vente, date_vente, valeur_fonciere, typeof(valeur_fonciere) AS type_prix FROM vente WHERE id_vente = 5")

,id_vente,date_vente,valeur_fonciere,type_prix
0,5,120000,2020-04-02,text


SQLite a tout accepté : un prix dans la colonne date, une date dans la colonne prix. **Après chaque import,
affiche quelques lignes** et vérifie qu'une date ressemble à une date et un prix à un nombre.

## 4. ⚠️ Piège : le format des dates

Les dates se comparent comme du texte, caractère par caractère. Seul le format **AAAA-MM-JJ** se trie dans l'ordre
du calendrier :

In [5]:
sql("""
SELECT
    '15/03/2020' BETWEEN '01/01/2020' AND '31/01/2020' AS format_francais,   -- 1 = vrai : faux résultat !
    '2020-03-15' BETWEEN '2020-01-01' AND '2020-01-31' AS format_iso          -- 0 = faux : juste
""")

,format_francais,format_iso
0,1,0


### ✍️ À toi de jouer

Sur ta base, écris une requête qui renvoie en **une seule ligne** : le nombre de biens, le nombre de biens sans code postal, et le nombre de biens sans numéro de voie.

In [6]:
# Ta réponse ici : sql(""" ... """)

<details>
<summary><b>💡 Voir une solution</b> (après avoir cherché !)</summary>

```sql
SELECT
    COUNT(*)                      AS nb_biens,
    COUNT(*) - COUNT(code_postal) AS sans_code_postal,
    COUNT(*) - COUNT(no_voie)     AS sans_numero
FROM bien;
```

`COUNT(*) - COUNT(colonne)` : le nombre de valeurs manquantes, à vérifier sur chaque colonne importante après un chargement.
</details>

## ✅ À retenir

1. Après un import, compter les lignes **des deux côtés** (CSV et base) : c'est la preuve demandée.
2. Un vide importé en `''` est lu comme 0 par `AVG` : il doit être `NULL`.
3. `COUNT(*)` compte les lignes, `COUNT(colonne)` les valeurs renseignées.
4. L'import se fait par position : afficher quelques lignes pour vérifier.
5. Dates au format AAAA-MM-JJ, sinon les comparaisons sont fausses.